# Autoencoder + Latent Dimension Scan (In/Out of Possession)

#### 1. Import Required Libraries

In [1]:
import os
import glob
import random
from typing import List, Tuple, Sequence, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)

#### Utility function

In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)
print("Device:", DEV)

Device: cpu


#### 2. Load and Explore the Dataset

In [3]:
CSV_PATTERN = "/home/falih/Mastercode/Bundesliga_2023_2024/formations/labled/data_understanding/cleaned_data_window_global.csv"

def load_data(csv_pattern: str) -> pd.DataFrame:
    files = glob.glob(csv_pattern)
    if not files:
        raise FileNotFoundError(f"No file found for pattern: {csv_pattern}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return df

df = load_data(CSV_PATTERN)
print("Loaded rows:", len(df))

Loaded rows: 488510


#### 3. Build X

In [4]:
PLAYER_FEATS = ["x_norm_mean", "y_norm_mean", "x_norm_std", "y_norm_std"]
GROUP_COLS   = ["match_id", "team_code", "window_global"]

def build_X(df: pd.DataFrame) -> Tuple[np.ndarray, pd.DataFrame]:
    need = set(PLAYER_FEATS + GROUP_COLS + ["player_name"])
    cols = [c for c in df.columns if c in need]
    df2 = df[cols].copy()

    rows, metas = [], []
    for keys, g in df2.groupby(GROUP_COLS, sort=False):
        if len(g) != 10:
            continue
        g_sorted = g.sort_values(["y_norm_mean", "x_norm_mean"], ascending=[False, True])
        rows.append(g_sorted[PLAYER_FEATS].to_numpy().reshape(-1))
        metas.append(keys)

    if not rows:
        raise ValueError("No valid windows with 10 outfield players found.")
    X = np.vstack(rows).astype("float32")
    meta = pd.DataFrame(metas, columns=GROUP_COLS)
    return X, meta

#### 4. Split by possession state

In [5]:
df_in  = df[df["dominant_possession_team"] == "In_possession"].copy()
df_out = df[df["dominant_possession_team"] == "Out_of_possession"].copy()

X_in,  meta_in  = build_X(df_in)
X_out, meta_out = build_X(df_out)

# Normalize
scaler_in  = StandardScaler()
Xn_in  = scaler_in.fit_transform(X_in).astype("float32")
scaler_out = StandardScaler()
Xn_out = scaler_out.fit_transform(X_out).astype("float32")


#### 5. Autoencoder architecture definition

In [6]:
class AE(nn.Module):
    def __init__(self, d_in: int = 40, d_z: int = 4):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(d_in, 64),
            nn.ReLU(),
            nn.Linear(64, d_z),
        )
        self.dec = nn.Sequential(
            nn.Linear(d_z, 64),
            nn.ReLU(),
            nn.Linear(64, d_in),
        )

    def forward(self, x):
        z = self.enc(x)
        xhat = self.dec(z)
        return xhat, z

def fit_autoencoder(
    Xn: np.ndarray,
    d_z: int = 4,
    epochs: int = 60,
    batch_size: int = 256,
    lr: float = 1e-3,
    verbose: bool = False,
) -> Tuple[nn.Module, np.ndarray, List[float]]:
    model = AE(d_in=Xn.shape[1], d_z=d_z).to(DEV)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    ds = TensorDataset(torch.from_numpy(Xn))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, pin_memory=(DEV.type=="cuda"))
    history = []

    model.train()
    for ep in range(epochs):
        loss_sum = 0.0
        for (xb,) in dl:
            xb = xb.to(DEV)
            xhat, _ = model(xb)
            loss = loss_fn(xhat, xb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            loss_sum += float(loss.item()) * xb.size(0)
        history.append(loss_sum / len(ds))

    model.eval()
    with torch.no_grad():
        Z = model.enc(torch.from_numpy(Xn).to(DEV)).cpu().numpy()

    return model, Z, history

#### 6. Latent dimension scan

In [8]:
LATENT_DIMS = (2, 3, 4, 5, 6, 8, 12)
K_LIST_DEFAULT = (5, 6, 7, 8, 9, 10, 11)

def kmeans_eval(Z: np.ndarray,
                k_list: Sequence[int] = K_LIST_DEFAULT,
                n_init: int = 10,
                repeats: int = 5) -> Dict:
    best_overall = None
    for k in k_list:
        best_sil = -1.0
        best_labels = None
        label_runs = []
        for _ in range(repeats):
            km = KMeans(n_clusters=k, n_init=n_init,
                        random_state=np.random.randint(0, 10_000))
            labels = km.fit_predict(Z)
            if len(set(labels)) < 2:
                continue
            sil = silhouette_score(Z, labels)
            if sil > best_sil:
                best_sil = sil
                best_labels = labels
            label_runs.append(labels)

        # stability via ARI
        aris = []
        for i in range(len(label_runs)):
            for j in range(i + 1, len(label_runs)):
                aris.append(adjusted_rand_score(label_runs[i], label_runs[j]))
        mean_ari = float(np.mean(aris)) if aris else np.nan

        if best_labels is not None:
            rec = {
                "k": k,
                "silhouette": best_sil,
                "davies_bouldin": davies_bouldin_score(Z, best_labels),
                "calinski_harabasz": calinski_harabasz_score(Z, best_labels),
                "stability_ari": mean_ari,
            }
            if (best_overall is None) or (rec["silhouette"] > best_overall["silhouette"]):
                best_overall = rec

    return best_overall

def scan_latent_dims(Xn: np.ndarray):
    for d in LATENT_DIMS:
        _, Z, history = fit_autoencoder(Xn, d_z=d, verbose=False)
        best = kmeans_eval(Z)
        print(
            f"[dz={d}] MSE={history[-1]:.4f} | "
            f"k={best['k']} | "
            f"Sil={best['silhouette']:.3f} | "
            f"DBI={best['davies_bouldin']:.3f} | "
            f"CH={best['calinski_harabasz']:.1f} | "
            f"ARI={best['stability_ari']:.3f}"
        )
print("IN POSSESSION")
scan_latent_dims(Xn_in)

print("OUT OF POSSESSION")
scan_latent_dims(Xn_out)

IN POSSESSION
[dz=2] MSE=0.6848 | k=5 | Sil=0.338 | DBI=0.913 | CH=13838.6 | ARI=0.959
[dz=3] MSE=0.6174 | k=5 | Sil=0.269 | DBI=1.072 | CH=10658.4 | ARI=0.979
[dz=4] MSE=0.5690 | k=6 | Sil=0.195 | DBI=1.344 | CH=6135.9 | ARI=0.971
[dz=5] MSE=0.5185 | k=5 | Sil=0.180 | DBI=1.518 | CH=5161.2 | ARI=0.966
[dz=6] MSE=0.4804 | k=5 | Sil=0.157 | DBI=1.685 | CH=4363.3 | ARI=0.995
[dz=8] MSE=0.4076 | k=5 | Sil=0.125 | DBI=2.000 | CH=3550.5 | ARI=0.977
[dz=12] MSE=0.3055 | k=5 | Sil=0.081 | DBI=2.461 | CH=2028.7 | ARI=0.965
OUT OF POSSESSION
[dz=2] MSE=0.6936 | k=6 | Sil=0.329 | DBI=0.913 | CH=13037.2 | ARI=0.933
[dz=3] MSE=0.6299 | k=7 | Sil=0.243 | DBI=1.091 | CH=7557.4 | ARI=0.957
[dz=4] MSE=0.5814 | k=7 | Sil=0.190 | DBI=1.303 | CH=5564.0 | ARI=0.970
[dz=5] MSE=0.5075 | k=7 | Sil=0.187 | DBI=1.397 | CH=4584.4 | ARI=0.989
[dz=6] MSE=0.4733 | k=6 | Sil=0.148 | DBI=1.735 | CH=4312.3 | ARI=0.958
[dz=8] MSE=0.4052 | k=7 | Sil=0.105 | DBI=2.015 | CH=2218.8 | ARI=0.968
[dz=12] MSE=0.3051 | k=5 | S